In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# ─── Data ────────────────────────────────────────────────────────
col_headers = ['Market', 'Preference', 'Confidence', 'Threshold Long', 'Threshold Short', 'Stop Loss', 'Return', 'Max DD', 'Sharpe', 'Win Rate', 'Pareto']
col_widths  = [0.07, 0.09, 0.09, 0.10, 0.10, 0.08, 0.10, 0.10, 0.08, 0.10, 0.09]

rows = [
    ['Thai', 'Balanced', '0.5628', '0.5331', '0.3000', '0.0403', '+25.35%',  '-43.09%', '0.13',  '46.67%', '40'],
    ['UK',   'Balanced', '0.5469', '0.5334', '0.3050', '0.1270', '-55.78%',  '-70.70%', '-0.10', '45.30%', '40'],
    ['Gold', 'Balanced', '0.5082', '0.5240', '0.3214', '0.0970', '+111.48%', '-25.82%', '0.30',  '48.19%', '40'],
    ['US',   'Balanced', '0.5081', '0.5397', '0.3542', '0.1450', '-98.96%',  '-99.29%', '-0.40', '41.50%', '40'],
    ['BTC',  'Balanced', '0.5044', '0.5455', '0.3146', '0.0331', '+646.82%', '-55.20%', '0.55',  '45.17%', '40'],
]

# ─── Layout ──────────────────────────────────────────────────────
ROW_H    = 0.62
HEADER_H = 0.72
FIG_W    = 18.0
N_ROWS   = len(rows)
FIG_H    = HEADER_H + N_ROWS * ROW_H + 0.20

BLACK = '#000000'
WHITE = '#ffffff'
GRAY  = '#f2f2f2'
LGRAY = '#cccccc'

market_bg = {
    'Thai': GRAY,
    'UK':   WHITE,
    'Gold': GRAY,
    'US':   WHITE,
    'BTC':  GRAY,
}

matplotlib.rcParams['font.family'] = ['DejaVu Sans']
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
ax.set_xlim(0, 1)
ax.set_ylim(0, FIG_H)
ax.axis('off')
fig.patch.set_facecolor(WHITE)

def cx(i):
    return sum(col_widths[:i])

# ── Header ────────────────────────────────────────────────────────
hdr_top = FIG_H - 0.10
hdr_bot = hdr_top - HEADER_H

ax.add_patch(mpatches.FancyBboxPatch(
    (0, hdr_bot), 1, HEADER_H,
    boxstyle='square,pad=0', facecolor=BLACK, edgecolor=BLACK,
    linewidth=2, clip_on=False))

for ci, (label, w) in enumerate(zip(col_headers, col_widths)):
    if ci > 0:
        ax.plot([cx(ci), cx(ci)], [hdr_bot, hdr_top],
                color=WHITE, linewidth=0.8, zorder=5)
    ax.text(cx(ci) + w / 2, (hdr_top + hdr_bot) / 2, label,
            ha='center', va='center',
            fontsize=11, fontweight='bold', color=WHITE)

ax.plot([0, 1], [hdr_bot, hdr_bot], color=BLACK, linewidth=2)

# ── Sub-header divider (Decision Variables vs Objectives) ────────
# Decision Vars: cols 2-5, Objectives: cols 6-9
# Just visual emphasis via thicker vertical lines at boundaries (col 6 = Return)

# ── Rows ──────────────────────────────────────────────────────────
prev_market = None
for ri, row_data in enumerate(rows):
    ry0 = hdr_bot - ri * ROW_H
    ry1 = ry0 - ROW_H
    market = row_data[0]

    ax.add_patch(mpatches.FancyBboxPatch(
        (0, ry1), 1, ROW_H,
        boxstyle='square,pad=0', facecolor=market_bg[market],
        edgecolor='none', clip_on=False))

    is_new = (market != prev_market)
    ax.plot([0, 1], [ry0, ry0],
            color=BLACK if is_new else LGRAY,
            linewidth=1.5 if is_new else 0.5)

    for ci, (val, w) in enumerate(zip(row_data, col_widths)):
        if ci > 0:
            # Thicker line between Decision Vars (col 5) and Objectives (col 6)
            line_w = 1.2 if ci == 6 else 0.6
            ax.plot([cx(ci), cx(ci)], [ry1, ry0],
                    color=BLACK if ci == 6 else LGRAY,
                    linewidth=line_w, zorder=5)

        if ci == 0:                          # Market — bold
            fw, fs, fst = 'bold', 11, 'normal'
        elif ci == 1:                        # Preference — italic
            fw, fs, fst = 'normal', 10.5, 'italic'
        elif ci in (2, 3, 4, 5):             # Decision variables — bold monospace style
            fw, fs, fst = 'bold', 10.5, 'normal'
        elif ci == 6:                        # Return — bold
            fw, fs, fst = 'bold', 11, 'normal'
        elif ci == 10:                       # Pareto size
            fw, fs, fst = 'normal', 10.5, 'normal'
        else:                                # Other metrics
            fw, fs, fst = 'normal', 10.5, 'normal'

        ax.text(cx(ci) + w / 2, (ry0 + ry1) / 2, val,
                ha='center', va='center',
                fontsize=fs, fontweight=fw, fontstyle=fst, color=BLACK)

    prev_market = market

# ── Border ───────────────────────────────────────────────────────
bot = hdr_bot - N_ROWS * ROW_H
ax.plot([0, 1], [bot, bot], color=BLACK, linewidth=2)
ax.plot([0, 0], [bot, hdr_top], color=BLACK, linewidth=2)
ax.plot([1, 1], [bot, hdr_top], color=BLACK, linewidth=2)

# ── Save ─────────────────────────────────────────────────────────
out_dir  = '/Users/oattao/project/p-e/ipynb/image'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'table8_moo_optimized_params.png')

plt.tight_layout(pad=0.1)
plt.savefig(out_path, dpi=200, bbox_inches='tight',
            facecolor=WHITE, pad_inches=0.12)
plt.show()
print(f'Saved → {out_path}')